# Train pairwise visual-bold fallback

Build one `(row_A_crop, row_B_crop, winner)` example per `visual_bold_lookup` question. The validation split is by `document_id`; no document appears in both train and validation. This notebook only writes a checkpoint for the ver1 fallback to load later.

In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
import json
import random
import re

import cv2
import numpy as np
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18

SEED = 20260813
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
EPOCHS = 12
LEARNING_RATE = 1e-3
VALIDATION_FRACTION = 0.20
HARD_GAP = 0.02
CELL_INSET_X = 0.04
CELL_INSET_Y = 0.12
CHECKPOINT_PATH = Path.cwd().resolve().parent / 'artifacts' / 'models' / 'bold_pair_resnet18.pt'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path.cwd().resolve().parent
DATA = PROJECT_ROOT / 'data' / 'training_set'
assert DATA.exists(), DATA
print(f'{DEVICE=}, {DATA=}, {NUM_WORKERS=}, {PIN_MEMORY=}')


DEVICE=device(type='cuda'), DATA=WindowsPath('E:/AIO/Project/DocViVQA/TACVU2/data/training_set')


In [2]:
def read_jsonl(path):
    with path.open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]


questions = read_jsonl(DATA / 'questions.jsonl')
labels_by_id = {item['question_id']: item for item in read_jsonl(DATA / 'labels.jsonl')}
manifests_by_id = {item['id']: item for item in read_jsonl(DATA / 'manifest.jsonl')}
annotations_by_key = {
    (item['document_id'], item['block_id']): item
    for item in read_jsonl(DATA / 'cell_annotations.jsonl')
}
visual_questions = [
    item for item in questions
    if labels_by_id[item['question_id']]['reasoning_type'] == 'visual_bold_lookup'
]
assert len(visual_questions) == 535


def center(block):
    x1, y1, x2, y2 = block['bbox']
    return (x1 + x2) / 2, (y1 + y2) / 2


def group_rows(blocks):
    grouped = defaultdict(list)
    for block in blocks:
        grouped[round(float(block['bbox'][1]), 6)].append(block)
    return [sorted(row, key=lambda block: center(block)[0]) for _, row in sorted(grouped.items())]


def table_blocks(blocks, table_index):
    titles = sorted(
        [block for block in blocks if block['bbox'][2] - block['bbox'][0] >= 0.65 and 'BẢNG' in str(block['text']).upper()],
        key=lambda block: block['bbox'][1],
    )
    if not titles:
        return blocks if table_index == 1 else []
    if not 1 <= table_index <= len(titles):
        return []
    start = titles[table_index - 1]['bbox'][1] - 1e-6
    end = titles[table_index]['bbox'][1] - 1e-6 if table_index < len(titles) else 1.0
    return [block for block in blocks if start <= center(block)[1] < end]


def header_block(blocks, text):
    matches = [block for block in blocks if str(block['text']) == text]
    return min(matches, key=lambda block: block['bbox'][1]) if matches else None


def cell_under(row, header):
    header_x = center(header)[0]
    matches = [block for block in row if block['bbox'][0] <= header_x <= block['bbox'][2]]
    return min(matches, key=lambda block: abs(center(block)[0] - header_x)) if matches else None


def matching_rows(blocks, conditions):
    headers = [header_block(blocks, column) for column, _ in conditions]
    if not conditions or any(header is None for header in headers):
        return []
    boundary = max(header['bbox'][3] for header in headers)
    matches = []
    for row in group_rows(blocks):
        if min(block['bbox'][1] for block in row) < boundary - 1e-6:
            continue
        cells = [cell_under(row, header) for header in headers]
        if all(cell is not None and str(cell['text']) == value for cell, (_, value) in zip(cells, conditions)):
            matches.append(row)
    return matches


VISUAL_PATTERNS = (
    re.compile(r'hãy chọn dòng in đậm giữa (?P<conditions>.+?), rồi đọc ô', re.I),
    re.compile(r'in đậm trong hai dòng (?P<conditions>.+?); giá trị ', re.I),
    re.compile(r'hai dòng (?P<conditions>.+?) có kiểu chữ khác nhau', re.I),
    re.compile(r'Giữa dòng (?P<conditions>.+?), dòng nào được in đậm', re.I),
)


def parse_visual_fields(question):
    location = re.search(r'bảng (\d+) ở trang (\d+)', question, re.I)
    match = next((pattern.search(question) for pattern in VISUAL_PATTERNS if pattern.search(question)), None)
    if location is None or match is None:
        return None
    text = match['conditions']
    pairs, previous_end = [], 0
    for quoted in re.finditer(r'“([^”]+)”', text):
        column = text[previous_end:quoted.start()].strip(' ,:.;')
        column = re.sub(r'^(?:và|với)\s+', '', column, flags=re.I)
        column = re.sub(r'^dòng(?:\s+có)?\s+', '', column, flags=re.I)
        if not column:
            return None
        pairs.append((column, quoted.group(1)))
        previous_end = quoted.end()
    split_at = next((index for index, pair in enumerate(pairs[1:], 1) if pair[0] == pairs[0][0]), None) if pairs else None
    if split_at is None:
        return None
    return int(location.group(1)), int(location.group(2)), [pairs[:split_at], pairs[split_at:]]


In [3]:
@lru_cache(maxsize=128)
def page_image(path):
    return np.asarray(Image.open(path).convert('L'))


def stroke_width(block, image):
    height, width = image.shape
    x1, y1, x2, y2 = block['bbox']
    crop = image[int(y1 * height):int(y2 * height), int(x1 * width):int(x2 * width)]
    if crop.size == 0 or min(crop.shape) < 5:
        return None
    _, ink = cv2.threshold(crop, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    border = max(1, round(min(ink.shape) * 0.03))
    ink[:border, :] = ink[-border:, :] = ink[:, :border] = ink[:, -border:] = 0
    if (ink > 0).sum() < 10:
        return None
    distance = cv2.distanceTransform(ink, cv2.DIST_L2, 3)
    return float(2 * distance[ink > 0].mean())


def row_stroke_score(row, image):
    scores = [score for block in row if (score := stroke_width(block, image)) is not None]
    return float(np.median(scores)) if scores else None


def row_is_bold(document_id, row):
    values = [annotations_by_key.get((document_id, block['block_id']), {}).get('is_bold') for block in row]
    return bool(values) and all(value is True for value in values)


def load_document_blocks(document_id):
    manifest = manifests_by_id[document_id]
    payload = json.loads((DATA / manifest['ocr_path']).read_text(encoding='utf-8'))
    return manifest, [block for page in payload['pages'] for block in page['blocks']]


pairs = []
for question in visual_questions:
    parsed = parse_visual_fields(question['question'])
    if parsed is None:
        continue
    table, page, condition_groups = parsed
    manifest, blocks = load_document_blocks(question['document_id'])
    page_blocks = [block for block in blocks if int(block['page']) == page]
    candidates = [matching_rows(table_blocks(page_blocks, table), group) for group in condition_groups]
    if any(len(rows) != 1 for rows in candidates):
        continue
    rows = [rows[0] for rows in candidates]
    bold = [row_is_bold(question['document_id'], row) for row in rows]
    if sum(bold) != 1:
        continue
    image_path = DATA / manifest['image_paths'][page - 1]
    image = page_image(image_path)
    scores = [row_stroke_score(row, image) for row in rows]
    pairs.append({
        'question_id': question['question_id'],
        'document_id': question['document_id'],
        'image_path': image_path,
        'rows': rows,
        'label': int(bold[1]),  # 0: A bold, 1: B bold
        'median_gap': float(abs(scores[0] - scores[1])) if all(score is not None for score in scores) else float('nan'),
    })

assert len(pairs) == 535, f'Expected 535 pairs, got {len(pairs)}'
print('pairs:', len(pairs), 'A winners:', sum(pair['label'] == 0 for pair in pairs), 'B winners:', sum(pair['label'] == 1 for pair in pairs))


pairs: 535 A winners: 260 B winners: 275


In [4]:
document_ids = sorted({pair['document_id'] for pair in pairs})
random.Random(SEED).shuffle(document_ids)
validation_ids = set(document_ids[:round(len(document_ids) * VALIDATION_FRACTION)])
train_pairs = [pair for pair in pairs if pair['document_id'] not in validation_ids]
validation_pairs = [pair for pair in pairs if pair['document_id'] in validation_ids]

assert {pair['document_id'] for pair in train_pairs}.isdisjoint({pair['document_id'] for pair in validation_pairs})
assert train_pairs and validation_pairs
print(f'train={len(train_pairs)}, validation={len(validation_pairs)}, validation documents={len(validation_ids)}')


def crop_row(image_path, row):
    image = Image.open(image_path).convert('L')
    width, height = image.size
    row_left = int(min(block['bbox'][0] for block in row) * width)
    row_top = int(min(block['bbox'][1] for block in row) * height)
    row_right = int(max(block['bbox'][2] for block in row) * width)
    row_bottom = int(max(block['bbox'][3] for block in row) * height)
    canvas = Image.new('L', (row_right - row_left, row_bottom - row_top), color=255)

    for block in row:
        x1, y1, x2, y2 = block['bbox']
        inset_x = (x2 - x1) * CELL_INSET_X
        inset_y = (y2 - y1) * CELL_INSET_Y
        left = int((x1 + inset_x) * width)
        top = int((y1 + inset_y) * height)
        right = int((x2 - inset_x) * width)
        bottom = int((y2 - inset_y) * height)
        if right <= left or bottom <= top:
            continue
        canvas.paste(image.crop((left, top, right, bottom)), (left - row_left, top - row_top))

    return canvas


def to_tensor(image):
    image = image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
    values = torch.from_numpy(np.asarray(image, dtype=np.float32) / 255.0)
    return ((values.unsqueeze(0).repeat(3, 1, 1)) - 0.5) / 0.5


class BoldPairDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        first, second = (to_tensor(crop_row(sample['image_path'], row)) for row in sample['rows'])
        return first, second, torch.tensor(sample['label'], dtype=torch.long), sample['median_gap']


loader_options = {'num_workers': NUM_WORKERS, 'pin_memory': PIN_MEMORY, 'persistent_workers': NUM_WORKERS > 0}
train_loader = DataLoader(BoldPairDataset(train_pairs), batch_size=BATCH_SIZE, shuffle=True, **loader_options)
validation_loader = DataLoader(BoldPairDataset(validation_pairs), batch_size=BATCH_SIZE, shuffle=False, **loader_options)


train=428, validation=107, validation documents=107


In [5]:
class PairResNet18(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = resnet18(weights=ResNet18_Weights.DEFAULT)
        self.encoder = nn.Sequential(*list(backbone.children())[:-1])
        for parameter in self.encoder.parameters():
            parameter.requires_grad = False
        self.head = nn.Linear(1024, 2)

    def forward(self, first, second):
        first_features = self.encoder(first).flatten(1)
        second_features = self.encoder(second).flatten(1)
        return self.head(torch.cat([first_features, second_features], dim=1))


model = PairResNet18().to(DEVICE)
optimizer = torch.optim.AdamW(model.head.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()
print('trainable parameters:', sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\minhtq/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 51.0MB/s]


trainable parameters: 2050


In [6]:
@torch.inference_mode()
def evaluate(loader):
    model.eval()
    records = []
    for first, second, labels, gaps in loader:
        probabilities = model(
            first.to(DEVICE, non_blocking=PIN_MEMORY),
            second.to(DEVICE, non_blocking=PIN_MEMORY),
        ).softmax(dim=1).cpu()
        for probability, label, gap in zip(probabilities, labels, gaps):
            records.append({
                'correct': int(probability.argmax().item() == label.item()),
                'confidence': float(probability.max().item()),
                'median_gap': None if torch.isnan(gap) else float(gap),
            })
    return records


for epoch in range(1, EPOCHS + 1):
    model.train()
    model.encoder.eval()  # Frozen backbone: do not update BatchNorm running statistics.
    total_loss = 0.0
    for first, second, labels, _ in train_loader:
        first = first.to(DEVICE, non_blocking=PIN_MEMORY)
        second = second.to(DEVICE, non_blocking=PIN_MEMORY)
        labels = labels.to(DEVICE, non_blocking=PIN_MEMORY)
        logits = model(first, second)
        loss = loss_fn(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
    validation = evaluate(validation_loader)
    accuracy = np.mean([row['correct'] for row in validation])
    print(f'epoch={epoch:02d} train_loss={total_loss / len(train_pairs):.4f} validation_accuracy={accuracy:.3f}')


epoch=01 train_loss=0.7724 validation_accuracy=0.505
epoch=02 train_loss=0.7360 validation_accuracy=0.495
epoch=03 train_loss=0.6478 validation_accuracy=0.561
epoch=04 train_loss=0.5392 validation_accuracy=0.673
epoch=05 train_loss=0.5164 validation_accuracy=0.701
epoch=06 train_loss=0.5283 validation_accuracy=0.692
epoch=07 train_loss=0.4414 validation_accuracy=0.673
epoch=08 train_loss=0.4597 validation_accuracy=0.720
epoch=09 train_loss=0.4267 validation_accuracy=0.720
epoch=10 train_loss=0.3897 validation_accuracy=0.692
epoch=11 train_loss=0.3794 validation_accuracy=0.729
epoch=12 train_loss=0.3720 validation_accuracy=0.776


In [7]:
validation = evaluate(validation_loader)
hard = [row for row in validation if row['median_gap'] is not None and row['median_gap'] < HARD_GAP]
print('validation accuracy:', np.mean([row['correct'] for row in validation]))
print('hard-subset accuracy:', np.mean([row['correct'] for row in hard]) if hard else float('nan'), f'({len(hard)} rows)')

for threshold in (0.50, 0.60, 0.70, 0.80, 0.90):
    selected = [row for row in hard if row['confidence'] >= threshold]
    coverage = len(selected) / len(hard) if hard else 0.0
    accuracy = np.mean([row['correct'] for row in selected]) if selected else float('nan')
    print(f'{threshold=:.2f} coverage={coverage:.3f} accuracy={accuracy:.3f} n={len(selected)}')


validation accuracy: 0.7757009345794392
hard-subset accuracy: 1.0 (2 rows)
threshold=0.50 coverage=1.000 accuracy=1.000 n=2
threshold=0.60 coverage=1.000 accuracy=1.000 n=2
threshold=0.70 coverage=1.000 accuracy=1.000 n=2
threshold=0.80 coverage=0.000 accuracy=nan n=0
threshold=0.90 coverage=0.000 accuracy=nan n=0


In [8]:
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    'architecture': 'pair_resnet18_frozen',
    'state_dict': model.state_dict(),
    'image_size': IMAGE_SIZE,
    'normalization': {'grayscale_to_rgb': True, 'mean': 0.5, 'std': 0.5},
    'crop_preprocessing': {'kind': 'cell_inset_composite', 'x_inset': CELL_INSET_X, 'y_inset': CELL_INSET_Y},
    'hard_gap': HARD_GAP,
    'seed': SEED,
    'validation_document_ids': sorted(validation_ids),
}, CHECKPOINT_PATH)
print('saved:', CHECKPOINT_PATH)


saved: E:\AIO\Project\DocViVQA\TACVU2\models\bold_pair_resnet18.pt
